<a href="https://colab.research.google.com/github/AMIRMOHAMMAD-OSS/NeurALPS-v2/blob/main/NeurALPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeurALPS


## Loading the models

In [ ]:
REPO_URL = "https://github.com/AMIRMOHAMMAD-OSS/NeurALPS-v2"
REVISION = "main"
from pathlib import Path
import subprocess, sys, json, hashlib, re
BASE = Path("/content/neuralps_work")
BASE.mkdir(exist_ok=True)
ROOT = BASE / "repository"
if not REPO_URL.startswith("https://github.com/") or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_./-]*", REVISION):
    raise ValueError("Provide a GitHub URL and valid revision")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(ROOT)], check=True)
subprocess.run(["git", "fetch", "origin", REVISION], cwd=ROOT, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT), "matplotlib", "huggingface-hub>=0.34,<1"], check=True)
sys.path.insert(0, str(ROOT))
print("Source commit:", SOURCE_COMMIT)
import numpy as np
from scipy.special import expit
from neuralps.head import load_model, predict
from neuralps.contracts import sha
HEAD = ROOT/"models/all494"
head_manifest = json.loads((HEAD/"manifest.json").read_text())
assert sha(HEAD/"supervised_head.npz") == head_manifest["model_sha256"]
print("Training rows:", head_manifest["trained_rows"], head_manifest["by_dataset"])
print("L2 penalty:", head_manifest["l2"], "— encoder updates:", head_manifest["encoder_updates"])
rows = json.loads((ROOT/"data/training_rows.json").read_text())
with np.load(ROOT/"data/training_features.npz", allow_pickle=False) as z:
    TRAIN_X = z["features"]
head = load_model(HEAD/"supervised_head.npz")
example_index = 40
if not 0 <= example_index < len(rows):
    raise ValueError("example_index must be 0–493")
print(rows[example_index])
print("In-sample supervised activity score:", float(expit(predict(head, TRAIN_X[example_index:example_index+1])[0])))

from neuralps.downloads import download, extract
RUNTIME_URL = "https://github.com/AMIRMOHAMMAD-OSS/NeurALPS-v2/releases/download/v0.1.0-20261009/NeurALPS_runtime_20261009.zip"
RUNTIME_SHA256 = "23ea8591e52a63f23d5ff8d22e01e58a60d75c0a9ffd1bc9622e167d9de6528f"
archive = download(RUNTIME_URL, BASE / "runtime.zip", RUNTIME_SHA256)
ASSETS = BASE / "runtime_assets"
if not ASSETS.exists():
    extract(archive, ASSETS)
manifest = json.loads((ASSETS / "manifest.json").read_text())
for name, digest in manifest["files_sha256"].items():
    target = (ASSETS / name).resolve()
    if not target.is_relative_to(ASSETS.resolve()) or sha(target) != digest:
        raise ValueError("Runtime checksum mismatch: " + name)
subprocess.run([sys.executable, str(ROOT / "scripts/verify_runtime.py"), "--assets", str(ASSETS), "--report", str(BASE / "encoder_parity.json")], check=True)
NATURAL_RELEASE_URL = "https://github.com/AMIRMOHAMMAD-OSS/NeurALPS-v2/releases/download/v0.2.0-20261010"
from neuralps.downloads import natural_reference, natural_repertoire
import urllib.error
try:
    reference = natural_reference(BASE, NATURAL_RELEASE_URL)
    print("Original natural-reference scale loaded and verified.")
except urllib.error.HTTPError as e:
    if e.code != 404:
        raise
    reference = None
    print("Natural-reference asset is not yet published. Scores will remain unscaled.")
    print("Run the one-time export and release upload in docs/EXPLORER_SETUP.md, then rerun this cell and the explorer cell.")

if "transformers" in sys.modules:
    raise RuntimeError("Restart the runtime before installing the preserved Transformers fork, then rerun setup")
wheel = ASSETS/manifest["transformers_distribution"]["wheel"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers==0.22.1", "safetensors>=0.4.3", "filelock", "packaging", "pyyaml", "regex", "requests", "tqdm", "huggingface-hub>=0.34,<1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", str(wheel)], check=True)
import torch
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("Select a BF16-capable GPU runtime for exact ESMC extraction. The encoder checks above work on CPU.")
print(torch.cuda.get_device_name(0))
LOCAL_ESMC_DIRECTORY = ""
from neuralps.embedding import download_esmc
ESMC = Path(LOCAL_ESMC_DIRECTORY) if LOCAL_ESMC_DIRECTORY else download_esmc(BASE/"ESMC-600M")
print("Pinned model folder:", ESMC)



Source commit: d269262663c607ee82462b2511d27bf2707cb604
Training rows: 494 {'bode1': 324, 'bode2': 105, 'tdomain': 65}
L2 penalty: 10.0 — encoder updates: 0
{'assembly_id': 'BODE_pIN241_pIN247_pIN286|assembly_line=1', 'dataset': 'bode1', 'observed_activity': 1}
In-sample supervised activity score: 0.7007634457413002
Original natural-reference scale loaded and verified.
Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

Pinned model folder: /content/neuralps_work/ESMC-600M


## Choose an annotated assembly
Choose **example** for a first test, or **upload JSON or saved result** to use your own assembly.

Your JSON file should contain:

- **`assembly_id`**: a name for the assembly.
- **`proteins`**: a list of proteins in biosynthetic order. Each protein needs a unique **`id`**, its full amino acid **`sequence`**, and a **`domains`** list.
- Each domain needs **`type`**, **`start`**, and **`end`**.

Use uppercase amino acid sequences without spaces or FASTA headers. For engineered assemblies, provide the final engineered sequence and its updated domain positions.

Coordinates are **0-based and end-exclusive**: residues 101–450 become `"start": 100, "end": 450`. Coordinates restart for each protein.

Use supported domain names: **`AMP-binding`** for A, **`PCP`** for T, **`Epimerization`** for E, and **`Thioesterase`** for TE. For C domains, specify the subtype, such as **`Condensation_LCL`**, **`Condensation_DCL`**, or **`Condensation_Dual`**.

Junctions are added automatically. After loading, select the region you want to score in the explorer.


In [ ]:
INPUT = "example" #@param ["example", "upload JSON or saved result"]
from google.colab import files
from neuralps.explorer import input_from_saved
if INPUT == "example":
    spec = json.loads((ROOT / "examples/AI_1_annotated.json").read_text())
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload one annotated assembly JSON or saved explorer file")
    spec = input_from_saved(next(iter(uploaded.values())).decode("utf-8"))
print("Assembly:", spec["assembly_id"])


Assembly: AI_1|context=minimal_GxpS_T3


## Explore an assembly line
Choose the **Start** and **End** of a region, then:

1. **Score selected segment** computes local and global compatibility with the region masked together.
2. **Find better replacements** searches matching fragments in the full natural repertoire: 34,927 assemblies. The bank downloads when first needed.
3. **Preview** shows a replacement in purple. **Retest in assembly** computes its new map and segment scores.

Switch between **Local**, **Global**, and **Supervised** above the map. Click a part to inspect its scores and natural-reference comparison.


In [ ]:
OUTPUTS = "pretrained only" #@param ["both", "pretrained only"]
from neuralps.explorer import ExplorerSession
session = ExplorerSession.from_spec(
    spec, ASSETS, ESMC,
    head_dir=HEAD if OUTPUTS == "both" else None,
    reference=reference,
    repertoire_factory=lambda: natural_repertoire(BASE, NATURAL_RELEASE_URL),
)
session.result["source_commit"] = SOURCE_COMMIT
result = session.result


ESMC: transformer_engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically — measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.
ESMC: neither xformers nor flash-attn is installed; falling back to PyTorch ``F.scaled_dot_product_attention``. The attention reduction order in bf16 differs from a fused kernel by ~1 bf16 ULP per attention block; compounded across the 80-block stack this reaches ~O(100) max-diff on the unnormalized residual stream. Install xformers (preferred) with `pip install xformers` for a fused attention kernel.
ESMC: flash-attn rotary kernel not installed; falling back to pure-PyTorch RoPE. For faster GPU inference run `pip install flash-attn`.


🚨 No checkpoint found for ESMCForSequenceClassification.forward. Please add a `checkpoint` arg to `auto_docstring` or add one in ESMCConfig's docstring
🚨 No checkpoint found for ESMCForTokenClassification.forward. Please add a `checkpoint` arg to `auto_docstring` or add one in ESMCConfig's docstring


### Open or reconnect the explorer
Run this cell again if the interface needs reconnecting. Your loaded model and computed scores are reused.


In [ ]:
from neuralps.explorer import show_explorer
show_explorer(session)


'neuralps.explorer.42a86c2b90c549f2'

## Save a result


In [ ]:
SAVE_COPY = True #@param {type:"boolean"}
if SAVE_COPY:
    from neuralps.explorer import save_explorer
    from neuralps.contracts import dump
    html_path = save_explorer(session, BASE / "neuralps_explorer.html")
    result_path = BASE / "neuralps_explorer.json"
    dump(result_path, session.payload())
    files.download(str(html_path))
    files.download(str(result_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>